In [1]:
import sympy as sp

In [2]:
from functools import lru_cache

@lru_cache(maxsize=None)
def get_ckn(k: int, n: int, p):
    if k<0 or k>n:
        return sp.Integer(0)
    if n==0:
        return sp.Integer(1) if k == 0 else sp.Integer(0)
    return get_ckn(k-1, n-1, p)/(2*p) + (k+1)*get_ckn(k+1, n-1, p)

In [3]:
#übungsaufe Fiboniacci rehe F(40) rekusiv implemtieren, einmal mit memoritzation ein ohne

In [4]:
x = sp.symbols('x', real = True)
alpha = sp.symbols('alpha', real = True, positive = True)
Ax = sp.symbols('Ax', real = True)

In [5]:
g0 = sp.exp(-alpha*(x-Ax)**2  )
g0

exp(-alpha*(-Ax + x)**2)

In [6]:
def get_hermite_gaussian(k):
    return g0.diff(Ax, k)


In [7]:
get_hermite_gaussian(20).simplify()

alpha**10*(1048576*alpha**10*(Ax - x)**20 - 99614720*alpha**9*(Ax - x)**18 + 3810263040*alpha**8*(Ax - x)**16 - 76205260800*alpha**7*(Ax - x)**14 + 866834841600*alpha**6*(Ax - x)**12 - 5721109954560*alpha**5*(Ax - x)**10 + 21454162329600*alpha**4*(Ax - x)**8 - 42908324659200*alpha**3*(Ax - x)**6 + 40226554368000*alpha**2*(Ax - x)**4 - 13408851456000*alpha*(Ax - x)**2 + 670442572800)*exp(-alpha*(Ax - x)**2)

In [8]:
nmax = 6
for n in range(nmax + 1):
    g = 0
    for k in range(0, n+1):
        g = g + get_ckn(k, n, alpha)*get_hermite_gaussian(k)
    g = g.simplify()
    print(f"g({n}) = {g}")

g(0) = exp(-alpha*(Ax - x)**2)
g(1) = (-Ax + x)*exp(-alpha*(Ax - x)**2)
g(2) = (Ax - x)**2*exp(-alpha*(Ax - x)**2)
g(3) = -Ax*(Ax - x)**2*exp(-alpha*(Ax - x)**2) + x*(Ax - x)**2*exp(-alpha*(Ax - x)**2)
g(4) = (Ax - x)**4*exp(-alpha*(Ax - x)**2)
g(5) = -Ax*(Ax - x)**4*exp(-alpha*(Ax - x)**2) + x*(Ax - x)**4*exp(-alpha*(Ax - x)**2)
g(6) = (Ax - x)**6*exp(-alpha*(Ax - x)**2)


## Implementierung der Überlappintegrale

In [9]:
alpha, beta = sp.symbols('alpha beta', real = True, positive = True)
Ax, Ay, Az = sp.symbols('Ax Ay Az', real = True)
Bx, By, Bz = sp.symbols('Bx By Bz', real = True)

p = alpha + beta
q = alpha*beta/p
RAB2 = (Ax-Bx)**2 + (Ay-By)**2 + (Az-Bz)**2

In [10]:
S00 = (sp.sqrt(sp.pi/p))**3 * sp.exp(-q*RAB2)
S00

pi**(3/2)*exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)/(alpha + beta))/(alpha + beta)**(3/2)

In [11]:
def build_derivative_table(base_integral, Lmax, Avars, Bvars):
    Ax, Ay, Az = Avars
    Bx, By, Bz = Bvars

    all_idx = [(i, j, L- i -j) for L in range(Lmax+1)
                                for i in range (L +1) 
                                for j in range(L + 1- i)]

    @lru_cache(maxsize=None)
    def derivative(i,j,k,l,m,n):
        if (i,j,k,l,m,n) == (0,0,0,0,0,0):
            return base_integral
        if i > 0:
            return sp.diff(derivative(i-1, j, k, l, m, n),Ax)
        if j > 0:
            return sp.diff(derivative(i, j-1, k, l, m, n),Ay)
        if k > 0:
            return sp.diff(derivative(i, j, k-1, l, m, n),Az)
        if l > 0:
            return sp.diff(derivative(i, j, k, l-1, m, n),Bx)
        if m > 0:
            return sp.diff(derivative(i, j, k, l, m-1, n),By)
        return sp.diff(derivative(i, j, k, l, m, n-1),Bz)

    derivatives_dict = {}
    for (i,j,k) in all_idx:
        for (l, m, n) in all_idx:
            derivatives_dict[(i,j,k,l,m,n)] = derivative(i,j,k,l,m,n)
    return derivatives_dict

In [12]:
Lmax = 2

derivatives_dict = build_derivative_table(S00, Lmax, (Ax, Ay, Az), (Bx, By, Bz))


In [13]:
for key, value in derivatives_dict.items():
    print(key, value)

(0, 0, 0, 0, 0, 0) pi**(3/2)*exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)/(alpha + beta))/(alpha + beta)**(3/2)
(0, 0, 0, 0, 0, 1) -pi**(3/2)*alpha*beta*(-2*Az + 2*Bz)*exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)/(alpha + beta))/(alpha + beta)**(5/2)
(0, 0, 0, 0, 1, 0) -pi**(3/2)*alpha*beta*(-2*Ay + 2*By)*exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)/(alpha + beta))/(alpha + beta)**(5/2)
(0, 0, 0, 1, 0, 0) -pi**(3/2)*alpha*beta*(-2*Ax + 2*Bx)*exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)/(alpha + beta))/(alpha + beta)**(5/2)
(0, 0, 0, 0, 0, 2) pi**(3/2)*alpha**2*beta**2*(-2*Az + 2*Bz)**2*exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)/(alpha + beta))/(alpha + beta)**(7/2) - 2*pi**(3/2)*alpha*beta*exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)/(alpha + beta))/(alpha + beta)**(5/2)
(0, 0, 0, 0, 1, 1) pi**(3/2)*alpha**2*beta**2*(-2*Ay + 2*By)*(-2*Az + 2*Bz)*exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2

In [14]:
def get_integral_expressions(integral_indices, derivatives_dict, alpha, beta):
    integral_expressions = {}
    for (i,j,k,l,m,n) in integral_indices:

        ci = [get_ckn(o, i, alpha) for o in range(i+1)]
        cj = [get_ckn(o, j, alpha) for o in range(j+1)]
        ck = [get_ckn(o, k, alpha) for o in range(k+1)]
        cl = [get_ckn(o, l, beta) for o in range(l+1)]
        cm = [get_ckn(o, m, beta) for o in range(m+1)]
        cn = [get_ckn(o, n, beta) for o in range(n+1)]

        expr = 0
        for o, co in enumerate(ci):
            for p, cp in enumerate(cj):
                for q, cq in enumerate(ck):

                    for r, cr in enumerate(cl):
                        for s, cs in enumerate(cm):
                            for t, ct in enumerate(cn):
                                expr += co*cp*cq*cr*cs*ct * derivatives_dict[(o, p, q, r, s ,t)]
        integral_expressions[(i, j, k, l, m, n)] = expr.simplify()
    return integral_expressions


In [15]:
def l_to_ijk(L):
    IJK = []
    for I in range(L, -1 , -1):
        for J in range(L - I, -1, -1):
            IJK.append((I, J, L - I - J))
    return sorted(IJK, reverse = True)

integral_indices = []
ijk = [t for L in range(Lmax +1) for t in l_to_ijk(L)]
for i in ijk:
    for j in ijk:
        integral_indices.append(i+j)
integral_indices

[(0, 0, 0, 0, 0, 0),
 (0, 0, 0, 1, 0, 0),
 (0, 0, 0, 0, 1, 0),
 (0, 0, 0, 0, 0, 1),
 (0, 0, 0, 2, 0, 0),
 (0, 0, 0, 1, 1, 0),
 (0, 0, 0, 1, 0, 1),
 (0, 0, 0, 0, 2, 0),
 (0, 0, 0, 0, 1, 1),
 (0, 0, 0, 0, 0, 2),
 (1, 0, 0, 0, 0, 0),
 (1, 0, 0, 1, 0, 0),
 (1, 0, 0, 0, 1, 0),
 (1, 0, 0, 0, 0, 1),
 (1, 0, 0, 2, 0, 0),
 (1, 0, 0, 1, 1, 0),
 (1, 0, 0, 1, 0, 1),
 (1, 0, 0, 0, 2, 0),
 (1, 0, 0, 0, 1, 1),
 (1, 0, 0, 0, 0, 2),
 (0, 1, 0, 0, 0, 0),
 (0, 1, 0, 1, 0, 0),
 (0, 1, 0, 0, 1, 0),
 (0, 1, 0, 0, 0, 1),
 (0, 1, 0, 2, 0, 0),
 (0, 1, 0, 1, 1, 0),
 (0, 1, 0, 1, 0, 1),
 (0, 1, 0, 0, 2, 0),
 (0, 1, 0, 0, 1, 1),
 (0, 1, 0, 0, 0, 2),
 (0, 0, 1, 0, 0, 0),
 (0, 0, 1, 1, 0, 0),
 (0, 0, 1, 0, 1, 0),
 (0, 0, 1, 0, 0, 1),
 (0, 0, 1, 2, 0, 0),
 (0, 0, 1, 1, 1, 0),
 (0, 0, 1, 1, 0, 1),
 (0, 0, 1, 0, 2, 0),
 (0, 0, 1, 0, 1, 1),
 (0, 0, 1, 0, 0, 2),
 (2, 0, 0, 0, 0, 0),
 (2, 0, 0, 1, 0, 0),
 (2, 0, 0, 0, 1, 0),
 (2, 0, 0, 0, 0, 1),
 (2, 0, 0, 2, 0, 0),
 (2, 0, 0, 1, 1, 0),
 (2, 0, 0, 1, 0, 1),
 (2, 0, 0, 0,

In [16]:
integral_exprssions = get_integral_expressions(integral_indices, derivatives_dict, alpha, beta)

In [17]:
for key,value in integral_exprssions.items():
    print(key, value)

(0, 0, 0, 0, 0, 0) pi**(3/2)*exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)/(alpha + beta))/(alpha + beta)**(3/2)
(0, 0, 0, 1, 0, 0) pi**(3/2)*alpha*(Ax - Bx)*exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)/(alpha + beta))/(alpha + beta)**(5/2)
(0, 0, 0, 0, 1, 0) pi**(3/2)*alpha*(Ay - By)*exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)/(alpha + beta))/(alpha + beta)**(5/2)
(0, 0, 0, 0, 0, 1) pi**(3/2)*alpha*(Az - Bz)*exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)/(alpha + beta))/(alpha + beta)**(5/2)
(0, 0, 0, 2, 0, 0) pi**(3/2)*(alpha*(2*alpha*beta*(Ax - Bx)**2 - alpha - beta) + (alpha + beta)**2)*exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)/(alpha + beta))/(2*beta*(alpha + beta)**(7/2))
(0, 0, 0, 1, 1, 0) pi**(3/2)*alpha**2*(Ax - Bx)*(Ay - By)*exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)/(alpha + beta))/(alpha + beta)**(7/2)
(0, 0, 0, 1, 0, 1) pi**(3/2)*alpha**2*(Ax - Bx)*(Az - Bz)*exp(-alpha*beta*(

In [18]:
Dx, Dy, Dz, P, Q, RAB2 = sp.symbols('Dx Dy Dz P Q RAB2')
subsdict = {Ax-Bx: Dx,
            Ay-By: Dy,
            Az-Bz: Dz,
            alpha+beta: P,
            alpha*beta: Q,
            (Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2: RAB2}

In [19]:
for key, value in integral_exprssions.items():
    integral_exprssions[key] = sp.simplify(value.subs(subsdict, simultaneous = True))

In [20]:
for key, value in integral_exprssions.items():
    print(f"Integral {key}: {value}")

Integral (0, 0, 0, 0, 0, 0): pi**(3/2)*exp(-Q*RAB2/P)/P**(3/2)
Integral (0, 0, 0, 1, 0, 0): pi**(3/2)*Dx*alpha*exp(-Q*RAB2/P)/P**(5/2)
Integral (0, 0, 0, 0, 1, 0): pi**(3/2)*Dy*alpha*exp(-Q*RAB2/P)/P**(5/2)
Integral (0, 0, 0, 0, 0, 1): pi**(3/2)*Dz*alpha*exp(-Q*RAB2/P)/P**(5/2)
Integral (0, 0, 0, 2, 0, 0): pi**(3/2)*(P**2 + alpha*(2*Dx**2*Q - P))*exp(-Q*RAB2/P)/(2*P**(7/2)*beta)
Integral (0, 0, 0, 1, 1, 0): pi**(3/2)*Dx*Dy*alpha**2*exp(-Q*RAB2/P)/P**(7/2)
Integral (0, 0, 0, 1, 0, 1): pi**(3/2)*Dx*Dz*alpha**2*exp(-Q*RAB2/P)/P**(7/2)
Integral (0, 0, 0, 0, 2, 0): pi**(3/2)*(P**2 + alpha*(2*Dy**2*Q - P))*exp(-Q*RAB2/P)/(2*P**(7/2)*beta)
Integral (0, 0, 0, 0, 1, 1): pi**(3/2)*Dy*Dz*alpha**2*exp(-Q*RAB2/P)/P**(7/2)
Integral (0, 0, 0, 0, 0, 2): pi**(3/2)*(P**2 + alpha*(2*Dz**2*Q - P))*exp(-Q*RAB2/P)/(2*P**(7/2)*beta)
Integral (1, 0, 0, 0, 0, 0): -pi**(3/2)*Dx*beta*exp(-Q*RAB2/P)/P**(5/2)
Integral (1, 0, 0, 1, 0, 0): pi**(3/2)*(-Dx**2*Q + P/2)*exp(-Q*RAB2/P)/P**(7/2)
Integral (1, 0, 0, 0, 1, 0

In [21]:
from sympy.printing.numpy import NumPyPrinter, _known_functions_numpy, _known_constants_numpy

In [22]:
printer = NumPyPrinter()
printer._module = "np"
printer.known_functions = {k: f"np.{v}" for k,v in _known_functions_numpy.items()}
printer.known_constants = {k: f"np.{v}" for k,v in _known_constants_numpy.items()}

In [23]:
print(printer.known_constants)

{'Exp1': 'np.e', 'Pi': 'np.pi', 'EulerGamma': 'np.euler_gamma', 'NaN': 'np.nan', 'Infinity': 'np.inf'}


In [24]:
def write_oneel_module(path, name = "S", integral_exprssions = None, parameter_list = None):
    lines = ["import numpy as np ",
             "from numba import njit",
             "@njit(cache = True, fastmath = True)",
             f"def {name}({parameter_list}):"]
    for key, value in integral_exprssions.items():
        lines.append(f"    if (i, j, k, l, m, n) == {key}:")
        lines.append(f"        return {printer.doprint(value)}")
    
    with open(path, "w", encoding= "utf-8") as f:
        f.write("\n".join(lines))


In [25]:
from pathlib import Path
path = Path.cwd() / "S.py"
write_oneel_module(path, name = "S", integral_exprssions= integral_exprssions, parameter_list= "Dx, Dy, Dz, P, Q, RAB2, alpha, beta")


In [26]:
Path.cwd()

WindowsPath('c:/Users/david/programmier/Master_Programmier')